# Cryptography Laboratory - Assignment 10
## MT19937 Stream Cipher and Predictable Token Attack

### Objective
Understand why statistical PRNGs (such as MT19937) are unsuitable for cryptographic applications by:
1. Implementing a toy stream cipher using MT19937.
2. Recovering a weak 16-bit seed using a known-plaintext attack.
3. Analyzing and attacking predictable timestamp-seeded reset tokens.


## Commit 1: Setup project structure and MT19937 initialization
- Import required libraries.
- Initialize the MT19937 random number generator.
- Set up seed handling and 8-bit keystream extraction.


In [ ]:
import random
import secrets
import string
import time
from typing import Optional

# Configuration: 16-bit seed search space (2^16 = 65,536)
SEED_16BIT_MAX = 65536

def init_mt19937(seed: int) -> random.Random:
    """Initializes the Mersenne Twister MT19937 PRNG with a specific integer seed."""
    return random.Random(seed)

def extract_keystream_byte(rng: random.Random) -> int:
    """Extracts one 8-bit keystream value (0-255) from MT19937."""
    return rng.getrandbits(8)

def generate_keystream(seed: int, length: int) -> bytes:
    """Generates length consecutive 8-bit keystream values from MT19937."""
    if length < 0:
        raise ValueError("Keystream length cannot be negative.")
    rng = init_mt19937(seed)
    return bytes(extract_keystream_byte(rng) for _ in range(length))

def validate_16bit_seed(seed: int) -> bool:
    """Validates whether an integer seed is within [0, 65535]."""
    return isinstance(seed, int) and 0 <= seed < SEED_16BIT_MAX

# Demonstration of Commit 1
sample_seed = 12345
sample_ks = generate_keystream(sample_seed, 16)
print(f"[*] Seed: {sample_seed}")
print(f"[*] Generated 16-byte keystream (hex): {sample_ks.hex()}")
print(f"[*] Seed validation check: {validate_16bit_seed(sample_seed)}")


## Commit 2: Implement MT19937 stream cipher
- Generate one 8-bit keystream value per plaintext byte.
- Implement XOR-based encryption ().
- Implement decryption using the same symmetric transformation ().
- Verify encryption and decryption using a test message.


In [ ]:
def mt19937_stream_cipher(data: bytes, seed: int) -> bytes:
    """MT19937 symmetric stream cipher engine for both encryption and decryption."""
    if isinstance(data, str):
        data = data.encode('utf-8')
    rng = init_mt19937(seed)
    return bytes(b ^ extract_keystream_byte(rng) for b in data)

def encrypt(plaintext: bytes, seed: int) -> bytes:
    """Encrypts plaintext bytes using the MT19937 stream cipher."""
    return mt19937_stream_cipher(plaintext, seed)

def decrypt(ciphertext: bytes, seed: int) -> bytes:
    """Decrypts ciphertext bytes using the MT19937 stream cipher."""
    return mt19937_stream_cipher(ciphertext, seed)

def verify_cipher_roundtrip(message: bytes, seed: int) -> bool:
    """Verifies that encryption followed by decryption recovers the exact original message."""
    ciphertext = encrypt(message, seed)
    recovered = decrypt(ciphertext, seed)
    return recovered == message

# Demonstration of Commit 2
test_message = b"Cryptography Laboratory: Assignment 10 Stream Cipher Verification!"
test_seed = 42424
ciphertext = encrypt(test_message, test_seed)
decrypted = decrypt(ciphertext, test_seed)

print(f"[*] Test Plaintext        : {test_message}")
print(f"[*] Ciphertext (hex)      : {ciphertext.hex()}")
print(f"[*] Decrypted Plaintext   : {decrypted}")
print(f"[*] Roundtrip verification: {decrypted == test_message}")
